# Notebook 03 — Evaluation & Analysis

Comprehensive evaluation of all trained ASL models:
- Per-class metrics (Precision, Recall, F1)
- Confusion matrices
- Bootstrap confidence intervals
- Statistical significance testing
- Ablation study
- Robustness analysis
- Model comparison plots


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd

from src.data.loader             import ASLDataLoader
from src.data.utils              import resize_images_rgb
from src.models.skeleton_extraction import extract_skeleton_features
from src.evaluation.metrics      import ModelEvaluator
from src.evaluation.visualizations import ResultVisualizer
from src.evaluation.ablation_study import AblationStudy
from src.evaluation.robustness_testing import RobustnessTester
from src.utils.config            import load_config

cfg = load_config('../config.yaml')
viz = ResultVisualizer(output_dir='../results/plots')

# Load test data
loader = ASLDataLoader(data_dir='../data/raw')
_, _, X_test, _, _, y_test = loader.load_dataset()
print(f'Test set: {X_test.shape}, Classes: {len(loader.class_names)}')

## 1. Load Saved Models

In [ ]:
models = {}
model_paths = {
    'Custom CNN'     : '../results/models/custom_cnn_best.h5',
    'MobileNetV2'    : '../results/models/mobilenet_best.h5',
    'EfficientNetB0' : '../results/models/efficientnet_best.h5',
    'Attention CNN'  : '../results/models/attention_cnn_best.h5',
    'Skeleton GCN'   : '../results/models/skeleton_gcn_best.h5',
}
for name, path in model_paths.items():
    try:
        models[name] = tf.keras.models.load_model(path)
        print(f'Loaded: {name}')
    except Exception as e:
        print(f'Could not load {name}: {e}')

## 2. Per-Model Evaluation

In [ ]:
# All models use 64x64 RGB images
test_inputs = {
    name: X_test for name in models
}
# Skeleton GCN uses extracted keypoints
X_test_skel = extract_skeleton_features(X_test)
test_inputs['Skeleton GCN'] = X_test_skel

all_reports = {}
for name, model in models.items():
    evaluator = ModelEvaluator(model, class_names=loader.class_names)
    report = evaluator.full_report(test_inputs[name], y_test)
    all_reports[name] = report
    print(f"{name}: Accuracy={report['accuracy']:.4f}  F1={report['f1_macro']:.4f}")


## 3. Model Comparison Table

In [ ]:
rows = []
for name, report in all_reports.items():
    ci_lo, ci_hi = report.get('bootstrap_ci', (None, None))
    rows.append({
        'Model'    : name,
        'Accuracy' : f"{report['accuracy']:.4f}",
        'Precision': f"{report['precision_macro']:.4f}",
        'Recall'   : f"{report['recall_macro']:.4f}",
        'F1'       : f"{report['f1_macro']:.4f}",
        '95% CI'   : f"[{ci_lo:.4f}, {ci_hi:.4f}]" if ci_lo else 'N/A',
    })

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
df.to_csv('../results/metrics/model_comparison.csv')

## 4. Confusion Matrices

In [ ]:
for name, model in models.items():
    evaluator = ModelEvaluator(model, class_names=loader.class_names)
    y_pred = np.argmax(model.predict(test_inputs[name], verbose=0), axis=1)
    viz.plot_confusion_matrix(
        y_test, y_pred,
        class_names=loader.class_names,
        title=f'Confusion Matrix — {name}',
        save_path=f'../results/plots/cm_{name.lower().replace(" ", "_")}.png'
    )
    plt.show()
    print(f'Saved confusion matrix for {name}')

## 5. Per-Class F1 Comparison

In [ ]:
viz.plot_model_comparison(
    all_reports,
    metric='f1_macro',
    save_path='../results/plots/model_comparison_f1.png'
)
plt.show()

## 6. Ablation Study

In [ ]:
ablation = AblationStudy(config_path='../config.yaml')
ablation_results = ablation.run(
    X_train=None,  # set to training data if re-training is desired
    X_test=X_test,
    y_test=y_test,
    saved_models=models
)
ablation.plot_results(ablation_results, save_path='../results/plots/ablation_study.png')
plt.show()

## 7. Robustness Testing

In [ ]:
tester = RobustnessTester()

best_model_name = max(all_reports, key=lambda k: all_reports[k]['accuracy'])
best_model = models[best_model_name]
print(f'Testing robustness of best model: {best_model_name}')

rob_results = tester.run_all_tests(
    model=best_model,
    X_test=test_inputs[best_model_name],
    y_test=y_test
)
tester.plot_results(rob_results, save_path='../results/plots/robustness.png')
plt.show()

## Summary

All evaluation results have been saved to `results/metrics/` and plots to `results/plots/`.
Proceed to **Notebook 04** for web deployment.
